# IMPORT THƯ VIỆN

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install pytorch_msssim

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import cv2
import matplotlib.pyplot as plt
from pytorch_msssim import ssim as ssim_fn
from skimage.metrics import structural_similarity as compare_ssim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# TIỀN XỬ LÝ DỮ LIỆU

In [ ]:
# ------------------------------
# Chuyển ảnh về ma trận ảnh Gray và resize
# ------------------------------
def matrix_images(image_folder, image_files, image_size=(224, 224)):
    matrix_img = []
    for img_file in image_files:
        img_path = os.path.join(image_folder, img_file)
        try:
            image = cv2.imread(img_path)
            if image is None:
                raise ValueError(f"Không thể đọc ảnh: {img_file}")
            image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
            # Resize ảnh
            image = cv2.resize(image, image_size)
            matrix_img.append(image)
        except Exception as e:
            print(f"Lỗi xử lý ảnh {img_file}: {e}")
    return matrix_img

In [ ]:
# ------------------------------
# Chuẩn hóa dữ liệu
# ------------------------------
def scaling_img(data_img):
    # Chuyển danh sách các mảng numpy thành một mảng numpy đa chiều
    data_array = np.array(data_img, dtype=np.float32)

    # Chuẩn hóa dữ liệu
    scaled_img = data_array / 255.0
    return scaled_img

In [ ]:
# ------------------------------
# Giảm nhiễu
# ------------------------------
def calculate_sigma(kernel_size):
    sigma = (kernel_size - 1) / 6.0
    return sigma

def reduce_noise(image, kernel_size=3):
    sigma = calculate_sigma(kernel_size)
    smoothed_image = cv2.GaussianBlur(image, (kernel_size, kernel_size), sigmaX=sigma)
    return smoothed_image

In [ ]:
def create_image_sequences(data, time_steps_input=3, time_steps_output=6):
    N, C, H, W = data.shape
    M = N - time_steps_input - time_steps_output + 1
    if M <= 0:
        raise ValueError("Không đủ khung hình để tạo chuỗi")
    X = np.stack([data[i:i+time_steps_input] for i in range(M)])
    Y = np.stack([data[i+time_steps_input:i+time_steps_input+time_steps_output] for i in range(M)])
    return X.astype(np.float32), Y.astype(np.float32)

In [ ]:
# ------------------------------
# Chia dữ liệu
# ------------------------------
def Split_Data(X_data, y_data):
    # Chia dữ liệu thành các tập train, val, test
    size = int(len(X_data) * 0.8)
    size_val = int((len(X_data) - size) / 2)

    X_train = X_data[:size]
    X_val = X_data[size:size + size_val]
    X_test = X_data[size + size_val:]

    y_train = y_data[:size]
    y_val = y_data[size:size + size_val]
    y_test = y_data[size + size_val:]

    return X_train, y_train, X_val, y_val, X_test, y_test

# XÂY DỰNG MÔ HÌNH Radar2Radar_GAN

### Hàm ConvLTSMCell

In [ ]:
class ConvLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size=(3,3), padding=(1,1), bias=True):
        super(ConvLSTMCell, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.kernel_size = kernel_size
        self.padding = padding

        # Convolution for input-to-state
        self.conv_x = nn.Conv2d(
            in_channels=self.input_dim,
            out_channels=self.hidden_dim * 4,
            kernel_size=self.kernel_size,
            padding=self.padding,
            bias=bias
        )
        # Convolution for hidden-state-to-state
        self.conv_h = nn.Conv2d(
            in_channels=self.hidden_dim,
            out_channels=self.hidden_dim * 4,
            kernel_size=self.kernel_size,
            padding=self.padding,
            bias=False
        )

    def forward(self, x, h, c):
        # Compute gates
        gates_x = self.conv_x(x)
        gates_h = self.conv_h(h)
        i, f, o, g = torch.chunk(gates_x + gates_h, chunks=4, dim=1)

        # Activation functions
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        o = torch.sigmoid(o)
        g = torch.tanh(g)

        # New cell and hidden states
        new_c = f * c + i * g
        new_h = o * torch.tanh(new_c)

        return new_h, new_c

### Mô hình CBAM

In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, ratio=16):
        super(ChannelAttention, self).__init__()
        self.in_channels = in_channels
        self.ratio = ratio
        # Global pooling to size 1x1
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        # MLP: 1x1 conv thay cho Dense
        mid_channels = in_channels // ratio
        self.fc1 = nn.Conv2d(in_channels, mid_channels, kernel_size=1, bias=True)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Conv2d(mid_channels, in_channels, kernel_size=1, bias=True)

        # Khởi tạo tương tự he_normal:
        nn.init.kaiming_normal_(self.fc1.weight, mode='fan_in', nonlinearity='relu')
        nn.init.constant_(self.fc1.bias, 0)
        nn.init.kaiming_normal_(self.fc2.weight, mode='fan_in', nonlinearity='relu')
        nn.init.constant_(self.fc2.bias, 0)

    def forward(self, x):
        # x: [B, C, H, W]
        # Avg pool path
        avg_out = self.avg_pool(x)                       # [B, C, 1, 1]
        avg_out = self.fc1(avg_out)                      # [B, C//ratio, 1, 1]
        avg_out = self.relu(avg_out)
        avg_out = self.fc2(avg_out)                      # [B, C, 1, 1]

        # Max pool path
        max_out = self.max_pool(x)                       # [B, C, 1, 1]
        max_out = self.fc1(max_out)
        max_out = self.relu(max_out)
        max_out = self.fc2(max_out)                      # [B, C, 1, 1]

        # Combine
        scale = torch.sigmoid(avg_out + max_out)         # [B, C, 1, 1]
        return x * scale                                 # broadcast multiply

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        self.kernel_size = kernel_size
        # Conv trên 2 kênh (avg + max) -> 1 kênh attention map
        padding = (kernel_size - 1) // 2
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, stride=1,
                              padding=padding, bias=False)
        # Khởi tạo he_normal
        nn.init.kaiming_normal_(self.conv.weight, mode='fan_in', nonlinearity='relu')

    def forward(self, x):
        # x: [B, C, H, W]
        # Tạo average map và max map theo kênh
        # avg: [B, 1, H, W], max: [B, 1, H, W]
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        # concat kênh: [B, 2, H, W]
        cat = torch.cat([avg_out, max_out], dim=1)
        # conv + sigmoid
        scale = torch.sigmoid(self.conv(cat))            # [B, 1, H, W]
        return x * scale                                 # broadcast multiply

class CBAM(nn.Module):
    def __init__(self, in_channels, ratio=16, kernel_size=7):
        super(CBAM, self).__init__()
        self.channel_attention = ChannelAttention(in_channels, ratio)
        self.spatial_attention = SpatialAttention(kernel_size)

    def forward(self, x):
        x = self.channel_attention(x)
        x = self.spatial_attention(x)
        return x

### Mô hình Generator

In [ ]:
class Seq2SeqUnrolled(nn.Module):
    def __init__(self, gf=64, in_steps=3, out_steps=6):
        super().__init__()
        self.gf = gf
        self.in_steps = in_steps
        self.out_steps = out_steps

        # --- ENCODER ---
        self.enc_conv1 = nn.Conv2d(1, gf, 3, padding=1)
        self.enc_cbam1 = CBAM(in_channels=gf, ratio=16, kernel_size=7)
        self.enc_bn1 = nn.BatchNorm2d(gf)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.enc_lstm1 = ConvLSTMCell(gf, gf)

        self.enc_conv2 = nn.Conv2d(gf, 2*gf, 3, padding=1)
        self.enc_cbam2 = CBAM(in_channels=2*gf, ratio=16, kernel_size=7)
        self.enc_bn2 = nn.BatchNorm2d(2*gf)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.enc_lstm2 = ConvLSTMCell(2*gf, 2*gf)

        self.enc_conv3 = nn.Conv2d(2*gf, 4*gf, 3, padding=1)
        self.enc_cbam3 = CBAM(in_channels=4*gf, ratio=16, kernel_size=7)
        self.enc_bn3 = nn.BatchNorm2d(4*gf)
        self.pool3 = nn.MaxPool2d(2, 2)
        self.enc_lstm3 = ConvLSTMCell(4*gf, 4*gf)

        self.enc_conv4 = nn.Conv2d(4*gf, 8*gf, 3, padding=1)
        self.enc_cbam4 = CBAM(in_channels=8*gf, ratio=16, kernel_size=7)
        self.enc_bn4 = nn.BatchNorm2d(8*gf)
        self.pool4 = nn.MaxPool2d(2, 2)
        self.enc_lstm4 = ConvLSTMCell(8*gf, 8*gf)

        # --- Skip‐conv layers ---
        self.skip_conv1 = nn.Conv2d(16*gf, 8*gf, 3, padding=1)
        self.skip_conv2 = nn.Conv2d( 8*gf, 4*gf, 3, padding=1)
        self.skip_conv3 = nn.Conv2d( 4*gf, 2*gf, 3, padding=1)
        self.skip_conv4 = nn.Conv2d( 2*gf,   gf, 3, padding=1)

        # --- DECODER with corrected ConvLSTMCell dims ---
        # Cấp 1: input_dim = 8*gf, hidden_dim = 8*gf
        self.dec_lstm1 = ConvLSTMCell(8*gf,  8*gf)
        self.dec_conv1 = nn.Conv2d(8*gf, 4*gf, 3, padding=1)
        self.dec_cbam1 = CBAM(in_channels=4*gf, ratio=16, kernel_size=7)
        self.dec_bn1   = nn.BatchNorm2d(4*gf)
        self.upsample1 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)

        # Cấp 2: input_dim = 4*gf, hidden_dim = 4*gf
        self.dec_lstm2 = ConvLSTMCell( 4*gf,  4*gf)
        self.dec_conv2 = nn.Conv2d(4*gf, 2*gf, 3, padding=1)
        self.dec_cbam2 = CBAM(in_channels=2*gf, ratio=16, kernel_size=7)
        self.dec_bn2   = nn.BatchNorm2d(2*gf)
        self.upsample2 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)

        # Cấp 3: input_dim = 2*gf, hidden_dim = 2*gf
        self.dec_lstm3 = ConvLSTMCell( 2*gf,  2*gf)
        self.dec_conv3 = nn.Conv2d(2*gf,   gf, 3, padding=1)
        self.dec_cbam3 = CBAM(in_channels=gf, ratio=16, kernel_size=7)
        self.dec_bn3   = nn.BatchNorm2d(gf)
        self.upsample3 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)

        # Cấp 4: input_dim =   gf, hidden_dim =   gf
        self.dec_lstm4 = ConvLSTMCell(gf, gf)
        self.dec_conv4 = nn.Conv2d(gf,     gf, 3, padding=1)
        self.dec_cbam4 = CBAM(in_channels=gf, ratio=16, kernel_size=7)
        self.dec_bn4   = nn.BatchNorm2d(gf)
        self.upsample4 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)

        self.final_conv1 = nn.Conv2d(gf, 2, kernel_size=1)
        self.final_conv2 = nn.Conv2d(2, 1, kernel_size=1)
        self.activation = nn.Sigmoid()
        self.leaky_relu = nn.LeakyReLU(0.2)

    def forward(self, x):
        # x: (batch, in_steps, 1, H, W)
        b, _, _, H, W = x.shape
        device = x.device

        # Init hidden states
        h1 = torch.zeros(b, self.gf, H//2, W//2, device=device)
        c1 = torch.zeros_like(h1)
        h2 = torch.zeros(b, 2*self.gf, H//4, W//4, device=device)
        c2 = torch.zeros_like(h2)
        h3 = torch.zeros(b, 4*self.gf, H//8, W//8, device=device)
        c3 = torch.zeros_like(h3)
        h4 = torch.zeros(b, 8*self.gf, H//16, W//16, device=device)
        c4 = torch.zeros_like(h4)

        # Encoder feature maps for skip
        skip1, skip2, skip3, skip4 = None, None, None, None

        # ENCODER
        for t in range(self.in_steps):
            xt = x[:, t]

            e1 = self.leaky_relu(self.enc_bn1(self.enc_cbam1(self.enc_conv1(xt))))
            e1p = self.pool1(e1) # 224 -> 112
            skip1 = e1p
            h1, c1 = self.enc_lstm1(e1p, h1, c1)

            e2 = self.leaky_relu(self.enc_bn2(self.enc_cbam2(self.enc_conv2(h1))))
            e2p = self.pool2(e2) # 112 -> 56
            skip2 = e2p
            h2, c2 = self.enc_lstm2(e2p, h2, c2)

            e3 = self.leaky_relu(self.enc_bn3(self.enc_cbam3(self.enc_conv3(h2))))
            e3p = self.pool3(e3) # 56 -> 28
            skip3 = e3p
            h3, c3 = self.enc_lstm3(e3p, h3, c3)

            e4 = self.leaky_relu(self.enc_bn4(self.enc_cbam4(self.enc_conv4(h3))))
            e4p = self.pool4(e4)  # 28 -> 14
            skip4 = e4p
            h4, c4 = self.enc_lstm4(e4p, h4, c4)

        # DECODER
        outputs = []
        prev = torch.zeros_like(h4)
        dh1, dc1 = h4, c4
        dh2, dc2 = h3, c3
        dh3, dc3 = h2, c2
        dh4, dc4 = h1, c1

        for _ in range(self.out_steps):
            # Level 1: bottom
            cat1 = self.skip_conv1(torch.cat([prev, skip4], dim=1))
            dh1, dc1 = self.dec_lstm1(cat1, dh1, dc1)
            y1 = self.upsample1(self.leaky_relu(self.dec_bn1(self.dec_cbam1(self.dec_conv1(dh1)))))  # 14 -> 28

            # Skip concat with skip4 (encoder e4) -> input to dec_lstm2
            cat2 = self.skip_conv2(torch.cat([y1, skip3], dim=1))
            dh2, dc2 = self.dec_lstm2(cat2, dh2, dc2)
            y2 = self.upsample2(self.leaky_relu(self.dec_bn2(self.dec_cbam2(self.dec_conv2(dh2)))))  # (b,2*gf,H/4,W/4)

            # Skip concat with skip3
            cat3 = self.skip_conv3(torch.cat([y2, skip2], dim=1))
            dh3, dc3 = self.dec_lstm3(cat3, dh3, dc3)
            y3 = self.upsample3(self.leaky_relu(self.dec_bn3(self.dec_cbam3(self.dec_conv3(dh3)))))  # (b,gf,H/2,W/2)

            # Skip concat with skip2
            cat4 = self.skip_conv4(torch.cat([y3, skip1], dim=1))
            dh4, dc4 = self.dec_lstm4(cat4, dh4, dc4)
            y4 = self.upsample4(self.leaky_relu(self.dec_bn4(self.dec_cbam4(self.dec_conv4(dh4)))))  # (b,gf,H,W)

            out_img = self.activation(self.final_conv2(self.final_conv1(y4)))
            outputs.append(out_img.unsqueeze(1))
            prev = dh1

        return torch.cat(outputs, dim=1)  # (b, out_steps, 1, H, W)


### Mô hình Discriminator

In [ ]:
class FrameDiscriminator(nn.Module):
    def __init__(self, in_channels=1, df=32):
        super().__init__()
        self.conv_blocks = nn.Sequential(
            nn.Conv2d(in_channels, df, 3, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm2d(df, momentum=0.8),
            nn.Conv2d(df, df*2, 3, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm2d(df*2, momentum=0.8),
            nn.Dropout2d(0.25),
            nn.Conv2d(df*2, df*4, 3, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm2d(df*4, momentum=0.8),
            nn.Dropout2d(0.25),
            nn.Conv2d(df*4, df*8, 3, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm2d(df*8, momentum=0.8),
            nn.Dropout2d(0.25)
        )
        self.validity = nn.Conv2d(df*8, 1, 3, padding=1)

    def forward(self, x):
        return self.validity(self.conv_blocks(x))

class SequenceDiscriminator(nn.Module):
    def __init__(self, seq_len, in_channels=1, df=32):
        super().__init__()
        self.seq_len = seq_len
        self.frame_disc = FrameDiscriminator(in_channels=in_channels, df=df)

    def forward(self, seq):
        # seq: (B, T, C, H, W)
        outs = [self.frame_disc(seq[:, t]) for t in range(self.seq_len)]
        return torch.stack(outs, dim=1)

### Mô hình Radar2Radar_GAN

In [ ]:
# --- Build Models ---
def build_radar2radar_gan(T_in, H, W, C, out_frame, feature_dim=32, device='cuda', lr_g=1e-4, lr_d=1e-4, label_smooth=0.9):
    # Generator and Discriminator
    gen = Seq2SeqUnrolled(gf=feature_dim, in_steps=T_in, out_steps=out_frame).to(device)
    disc = SequenceDiscriminator(seq_len=out_frame, in_channels=C, df=feature_dim).to(device)

    # Optimizers
    opt_g = optim.Adam(gen.parameters(), lr=lr_g)
    opt_d = optim.Adam(disc.parameters(), lr=lr_d)

    return gen, disc, opt_g, opt_d, (H//16, W//16), label_smooth

In [ ]:
# --- Combined GAN Module ---
class CombinedGAN(nn.Module):
    def __init__(self, generator, discriminator):
        super().__init__()
        self.generator = generator
        self.discriminator = discriminator

    def forward(self, x):
        fake_y = self.generator(x)
        validity = self.discriminator(fake_y)
        return validity, fake_y

### Hàm tạo bộ huấn luyện mô hình

In [ ]:
# --- GAN Loss Function với SSIM + MSE ---
def compute_gan_loss_with_ssim(combined_gan, x, y, adv_crit, rec_crit,
                               adv_w=1.0, rec_w=100.0, ssim_w=10.0, label_smooth=0.9):

    validity, fake_y = combined_gan(x)
    valid = torch.full_like(validity, label_smooth)
    adv_loss = adv_crit(validity, valid)

    # --- Reconstruction loss: MSE ---
    rec_loss = rec_crit(fake_y, y)

    # --- SSIM loss ---
    B, T_out, C, H, W = fake_y.shape
    ssim_sum = 0.0
    for t in range(T_out):
        fake_frame = fake_y[:, t]  # (B, C, H, W)
        true_frame = y[:, t]       # (B, C, H, W)
        ssim_val = ssim_fn(fake_frame, true_frame, data_range=1.0, size_average=True)
        ssim_sum += ssim_val

    ssim_avg = ssim_sum / T_out  # vẫn là tensor scalar
    loss_ssim = 1.0 - ssim_avg    # càng nhỏ càng tốt

    # --- Tổng hợp Generator loss ---
    loss_g = adv_w * adv_loss + rec_w * rec_loss + ssim_w * loss_ssim

    return loss_g

In [ ]:
class Radar2RadarTrainer:
    def __init__(self, generator, discriminator, opt_g, opt_d, device, label_smooth):
        self.generator = generator.to(device)
        self.discriminator = discriminator.to(device)
        self.combined = CombinedGAN(self.generator, self.discriminator).to(device)
        self.opt_g, self.opt_d = opt_g, opt_d
        self.adv_crit = nn.MSELoss()
        self.rec_crit = nn.L1Loss()
        self.device = device
        self.label_smooth = label_smooth

    def train_epoch(self, loader, disc_patch):
        self.generator.train()
        self.discriminator.train()
        sum_d, sum_g = 0.0, 0.0
        for x, y in loader:
            x, y = x.to(self.device), y.to(self.device)
            bs = x.size(0)
            T_out = y.size(1)
            valid = torch.full((bs, T_out, 1, *disc_patch), self.label_smooth, device=self.device)
            fake = torch.zeros_like(valid)

            # D update
            fake_y = self.generator(x).detach()
            real_in = y + 0.1 + torch.randn_like(y)
            fake_in = fake_y + 0.1 + torch.randn_like(fake_y)

            self.opt_d.zero_grad()
            d_real = self.discriminator(real_in)
            d_fake = self.discriminator(fake_in)
            loss_d = 0.5 * (self.adv_crit(d_real, valid) + self.adv_crit(d_fake, fake))
            loss_d.backward()
            self.opt_d.step()
            sum_d += loss_d.item()

            # G update
            for _ in range(2):
                self.opt_g.zero_grad()
                loss_g = compute_gan_loss_with_ssim(
                    self.combined, x, y,
                    adv_crit=self.adv_crit,
                    rec_crit=self.rec_crit,
                )
                loss_g.backward()
                self.opt_g.step()
                sum_g += loss_g.item()

        return sum_d / len(loader), sum_g / len(loader) / 2

    @torch.no_grad()
    def eval_epoch(self, loader):
        self.generator.eval()
        self.discriminator.eval()
        sum_v = 0.0
        for x, y in loader:
            x, y = x.to(self.device), y.to(self.device)
            loss_g = compute_gan_loss_with_ssim(
                self.combined, x, y,
                adv_crit=self.adv_crit,
                rec_crit=self.rec_crit,
            )
            sum_v += loss_g.item()
        return sum_v / len(loader)

In [ ]:
def train_radar2radar_gan(train_A, train_B, val_A, val_B, batch_size=4, epochs=100, save_path="best_gen.pth", device="cuda"):
    # prepare data
    N, T_in, H, W, C = train_A.shape
    T_out = train_B.shape[1]
    to_t = lambda arr: torch.from_numpy(arr).permute(0, 1, 4, 2, 3).float()
    ds_train = TensorDataset(to_t(train_A), to_t(train_B))
    ds_val = TensorDataset(to_t(val_A), to_t(val_B))
    loader_tr = DataLoader(ds_train, batch_size)
    loader_va = DataLoader(ds_val, batch_size)

    # build
    gen, disc, opt_g, opt_d, disc_patch, ls = build_radar2radar_gan(T_in, H, W, C, T_out, device=device)
    trainer = Radar2RadarTrainer(gen, disc, opt_g, opt_d, device, label_smooth=ls)

    best_v = float("inf")
    d_losses, g_losses, v_losses = [], [], []

    for ep in range(1, epochs + 1):
        # Đúng:
        d_l, g_l = trainer.train_epoch(loader_tr, disc_patch)
        v_l = trainer.eval_epoch(loader_va)

        d_losses.append(d_l)
        g_losses.append(g_l)
        v_losses.append(v_l)

        print(f"Epoch {ep}/{epochs} | D_loss:{d_l:.4f} G_loss:{g_l:.4f} G_val_loss:{v_l:.4f}")
        if v_l < best_v:
            best_v = v_l
            torch.save(gen.state_dict(), save_path)
            print(f"✔️ Saved epoch {ep}")

    print("Training finished.")

    return d_losses, g_losses, v_losses

### Hàm dự đoán

In [ ]:
def predict_radar2radar(generator, X, batch_size=1, model_path=None, device=None):
    # Thiết bị
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    else:
        device = torch.device(device)

    # Load weights nếu có
    if model_path is not None:
        sd = torch.load(model_path, map_location=device)
        # Nếu state_dict có 'module.' prefix (khi train DataParallel), ta có thể strip:
        try:
            generator.load_state_dict(sd)
        except RuntimeError:
            from collections import OrderedDict
            new_state = OrderedDict()
            for k, v in sd.items():
                name = k.replace("module.", "")
                new_state[name] = v
            generator.load_state_dict(new_state)

    generator.to(device).eval()

    # Chuẩn bị TensorDataset
    # Nếu X là numpy array:
    if isinstance(X, np.ndarray):
        X_tensor = torch.from_numpy(X.astype(np.float32))
    elif isinstance(X, torch.Tensor):
        X_tensor = X.detach().cpu()
    else:
        raise ValueError("X phải là numpy array hoặc torch.Tensor")

    # X_tensor có thể shape (N, T_in, H, W, C) hoặc (N, T_in, C, H, W)
    if X_tensor.ndim == 5:
        N, T, d2, d3, d4 = X_tensor.shape
        # Trường hợp channel last: shape[-1] in (1,3)
        if X_tensor.shape[-1] in (1, 3):
            # (N, T, H, W, C) -> permute về (N, T, C, H, W)
            X_tensor = X_tensor.permute(0, 1, 4, 2, 3).contiguous()
        # Trường hợp channel-first: shape[2] in (1,3)
        elif X_tensor.shape[2] in (1, 3):
            # Giữ nguyên (N, T, C, H, W)
            pass
        else:
            raise ValueError(f"X_tensor có shape không mong đợi: {X_tensor.shape}. "
                             "Vui lòng đưa về (N, T_in, H, W, C) hoặc (N, T_in, C, H, W) với C=1 hoặc 3.")
    else:
        raise ValueError(f"X_tensor có shape không mong đợi: {X_tensor.shape}. "
                         "Phải là 5D.")

    ds = TensorDataset(X_tensor)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)

    preds_list = []
    with torch.no_grad():
        for (Xb,) in loader:
            # Xb shape (B, T_in, C, H, W)
            Xb = Xb.to(device)
            out = generator(Xb)  # mong shape (B, T_out, 1, H, W) hoặc (B, T_out, C, H, W)
            out_np = out.cpu().detach().numpy()
            # Đưa về dạng (B, T_out, H, W, C) nếu C=1 hoặc 3
            if out_np.ndim == 5:
                B, T_out, C, H, W = out_np.shape
                out_np = out_np.transpose(0, 1, 3, 4, 2)  # (B, T_out, H, W, C)
                if C == 1:
                    out_np = out_np[..., 0]  # (B, T_out, H, W)
            else:
                raise ValueError(f"Output generator có shape không mong đợi: {out_np.shape}")
            preds_list.append(out_np)

    preds = np.concatenate(preds_list, axis=0)
    return preds

# ĐÁNH GIÁ MÔ HÌNH VÀ TRỰC QUAN KẾT QUẢ

### Đánh giá kết quả

In [ ]:
def evaluate_radar2radar(preds, Y, data_range=1.0):
    # Chuyển Y về numpy nếu cần
    if isinstance(Y, torch.Tensor):
        Y_np = Y.detach().cpu().numpy()
    elif isinstance(Y, np.ndarray):
        Y_np = Y
    else:
        raise ValueError("Y phải là numpy array hoặc torch.Tensor")

    preds_np = preds
    if preds_np.shape != Y_np.shape:
        # Nếu Y có shape (N, T_out, H, W, 1), ta có thể squeeze:
        if Y_np.ndim == 5 and Y_np.shape[-1] == 1 and preds_np.shape == Y_np[..., 0].shape:
            Y_np = Y_np[..., 0]
        else:
            raise ValueError(f"Shape preds {preds_np.shape} và Y {Y_np.shape} không khớp")

    N = preds_np.shape[0]
    # Flatten channel nếu cần: (N, T_out, H, W, C) -> (N, T_out, H, W)
    if preds_np.ndim == 5:
        # C>1 hoặc C=1: chúng ta trung bình qua channel
        preds_flat = preds_np.mean(axis=-1)
        Y_flat = Y_np.mean(axis=-1)
    else:
        preds_flat = preds_np
        Y_flat = Y_np

    _, T_out, H, W = preds_flat.shape
    sum_mae  = np.zeros(T_out, dtype=np.float64)
    sum_mse  = np.zeros(T_out, dtype=np.float64)
    sum_ssim = np.zeros(T_out, dtype=np.float64)

    for i in range(N):
        for t in range(T_out):
            p = preds_flat[i, t]
            g = Y_flat[i, t]
            sum_mae[t]  += np.mean(np.abs(p - g))
            sum_mse[t]  += np.mean((p - g) ** 2)
            try:
                s = compare_ssim(g, p, data_range=data_range)
            except ValueError:
                s = 1.0 if np.allclose(p, g) else 0.0
            sum_ssim[t] += s

    mae_per_frame  = sum_mae / N
    mse_per_frame  = sum_mse / N
    rmse_per_frame = np.sqrt(mse_per_frame)
    ssim_per_frame = sum_ssim / N

    # In kết quả theo frame
    print(f"Đánh giá cho từng frame:")
    for t in range(T_out):
        print(f" Frame {t+1}: MSE={mse_per_frame[t]:.6f}, "
              f"MAE={mae_per_frame[t]:.6f}, RMSE={rmse_per_frame[t]:.6f}, "
              f"SSIM={ssim_per_frame[t]:.4f}")

    # Tính và in trung bình tổng thể
    mae_all  = np.mean(mae_per_frame)
    mse_all  = np.mean(mse_per_frame)
    rmse_all = np.mean(rmse_per_frame)
    ssim_all = np.mean(ssim_per_frame)

    print("\n=== Trung bình tổng thể qua tất cả frames ===")
    print(f"MSE:  {mse_all:.6f}")
    print(f"MAE:  {mae_all:.6f}")
    print(f"RMSE: {rmse_all:.6f}")
    print(f"SSIM: {ssim_all:.4f}")

    results = {
        'MAE_per_frame': mae_per_frame,
        'MSE_per_frame': mse_per_frame,
        'RMSE_per_frame': rmse_per_frame,
        'SSIM_per_frame': ssim_per_frame,
        'MAE_all': mae_all,
        'MSE_all': mse_all,
        'RMSE_all': rmse_all,
        'SSIM_all': ssim_all,
        'count': N
    }
    return results

### Trực quan hóa ảnh dự báo

In [ ]:
def visualize_radar2radar_preds(X, Y, preds, num_samples=4, indices=None):
    # Chuyển X, Y, preds thành numpy nếu là Tensor
    if isinstance(X, torch.Tensor):
        X_np = X.detach().cpu().numpy()
    else:
        X_np = X
    if isinstance(Y, torch.Tensor):
        Y_np = Y.detach().cpu().numpy()
    else:
        Y_np = Y
    preds_np = preds

    N = preds_np.shape[0]

    def flatten_channel(arr):
        # Trả về (N, T, H, W)
        if arr.ndim == 5:
            # Có thể (N, T, H, W, C) hoặc (N, T, C, H, W)
            if arr.shape[-1] in (1, 3):
                arr2 = arr  # (N, T, H, W, C)
            else:
                # (N, T, C, H, W) -> (N, T, H, W, C)
                arr2 = arr.transpose(0, 1, 3, 4, 2)
            C = arr2.shape[-1]
            if C == 1:
                return arr2[..., 0]
            else:
                return arr2.mean(axis=-1)
        elif arr.ndim == 4:
            # (N, T, H, W)
            return arr
        else:
            raise ValueError(f"Shape không mong đợi: {arr.shape}")

    Xf = flatten_channel(X_np)
    Yf = flatten_channel(Y_np)
    Pf = flatten_channel(preds_np)

    # Chọn indices
    if indices is None:
        if num_samples > N:
            num_samples = N
        indices = np.random.choice(N, size=num_samples, replace=False)
    else:
        indices = [i for i in indices if 0 <= i < N]
        if len(indices) == 0:
            raise ValueError("Không có index hợp lệ để hiển thị")
        num_samples = len(indices)

    for idx in indices:
        xin = Xf[idx]  # (T_in, H, W)
        ygt = Yf[idx]  # (T_out, H, W)
        ypr = Pf[idx]  # (T_out, H, W)
        T_in = xin.shape[0]
        T_out = ygt.shape[0]
        cols = max(T_in, T_out)
        fig, axes = plt.subplots(3, cols, figsize=(cols*3, 3*3))
        # Hàng Input
        for t in range(cols):
            ax = axes[0, t] if cols > 1 else axes[0]
            if t < T_in:
                img = xin[t]
                ax.imshow(img)
                ax.set_title(f'Input t={t}')
            ax.axis('off')
        axes[0,0].set_ylabel('Input')
        # Hàng GT
        for t in range(cols):
            ax = axes[1, t] if cols > 1 else axes[1]
            if t < T_out:
                img = ygt[t]
                ax.imshow(img)
                ax.set_title(f'GT t={t}')
            ax.axis('off')
        axes[1,0].set_ylabel('Ground Truth')
        # Hàng Pred
        for t in range(cols):
            ax = axes[2, t] if cols > 1 else axes[2]
            if t < T_out:
                img = ypr[t]
                ax.imshow(img)
                ax.set_title(f'Pred t={t}')
            ax.axis('off')
        axes[2,0].set_ylabel('Predicted')
        plt.tight_layout()
        plt.show()

# THỰC HIỆN CHẠY MÔ HÌNH

### Đọc dữ liệu

In [ ]:
# Đọc từng file ảnh và sắp xếp theo thứ tự tên tệp
image_folder = '/content/drive/MyDrive/BacSon/processed_data_img'
image_files = sorted([f for f in os.listdir(image_folder) if os.path.isfile(os.path.join(image_folder, f))],
                     key=lambda x: int(''.join(filter(str.isdigit, x))) if any(c.isdigit() for c in x) else x)

In [ ]:
# params
timesteps = 4
out_timesteps = 6
im_width = im_height = 224
channels = 1

In [ ]:
# Resize và chuyển đổi kênh màu
data_img = matrix_images(image_folder, image_files[-64:], image_size=(im_width, im_height))

if len(data_img) == 0:
    raise ValueError("Không có ảnh nào được xử lý, vui lòng kiểm tra lại thư mục!")

# Thực hiện chuẩn hóa và khử nhiễu dữ liệu
data_img_processing = []
data_img_scaled = scaling_img(data_img)
for img in data_img_scaled:
    data_noise_img = reduce_noise(img, kernel_size=3)
    data_img_processing.append(data_noise_img)

# Chuyển thành numpy array và thêm trục kênh
data_image = np.expand_dims(np.array(data_img_processing), axis=-1)

# Tạo chuỗi dữ liệu ảnh liên tiếp
datas, labels = create_image_sequences(data_image, time_steps_input=timesteps, time_steps_output=out_timesteps)

# Chia dữ liệu
X_train, y_train, X_val, y_val, X_test, y_test = Split_Data(datas, labels)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

### Thực hiện huấn luyện mô hình

In [ ]:
# 20p
d_losses, g_losses, v_losses = train_radar2radar_gan(
    X_train, y_train,
    X_val, y_val,
    epochs=200,
    batch_size=4,
    save_path='/content/drive/MyDrive/BacSon/ISI-2025/model_SR2-GAN_CBAM_v6.pth'
    )

In [ ]:
# --- Plot losses ---
plt.figure(figsize=(10, 5))
plt.plot(d_losses, label="D_loss")
plt.plot(g_losses, label="G_loss")
plt.plot(v_losses, label="G_val_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss Curve")
plt.legend()
plt.tight_layout()
plt.savefig("loss_curve.png")
print("📈 Saved loss plot to loss_curve.png")

### Kiểm thử mô hình và đánh giá mô hình

In [ ]:
model_path = '/content/drive/MyDrive/BacSon/ISI-2025/model_SR2-GAN_CBAM_v6.pth'

generator, _, _, _, _, _ = build_radar2radar_gan(4, 224, 224, 1, 6)

preds = predict_radar2radar(generator, X_test,
                            batch_size=4,
                            model_path=model_path,
                            device='cuda')

In [ ]:
results = evaluate_radar2radar(preds, y_test, data_range=1.0)

In [ ]:
visualize_radar2radar_preds(X_test, y_test, preds, num_samples=6)